In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_log_error

In [ ]:
# Load datasets
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

train.head()

In [ ]:
# Select target and drop non necessary variables
X = train.drop(columns=["Rings"])
y = train["Rings"]

test_ids = test["id"]

X = X.drop(columns=["id"])
test_clean = test.drop(columns=["id"])

In [ ]:
# Train validation split
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Lasso regression pipeline L1
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_log_error

categorical_features = ["Sex"]
numeric_features = [col for col in X.columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

lasso_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Lasso(alpha=0.001, max_iter=10000, random_state=42))
    ]
)

In [ ]:
# Train Lasso Model
lasso_model.fit(X_train, np.log1p(y_train))

In [ ]:
# Model Validation
valid_preds_log = lasso_model.predict(X_valid)
valid_preds = np.expm1(valid_preds_log)

valid_preds = np.clip(valid_preds, 1, None)

rmsle = np.sqrt(mean_squared_log_error(y_valid, valid_preds))

print("Lasso RMSLE:", rmsle)

In [ ]:
# View Selected Features
onehot_names = lasso_model.named_steps["preprocessor"] \
    .named_transformers_["cat"] \
    .get_feature_names_out(categorical_features)

feature_names = list(onehot_names) + numeric_features

coefficients = lasso_model.named_steps["model"].coef_

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute Coefficient": abs(coefficients)
}).sort_values(by="Absolute Coefficient", ascending=False)

coef_df

In [ ]:
# Predict on Test dataset
test_preds_log = lasso_model.predict(test_clean)
test_preds = np.expm1(test_preds_log)

test_preds = np.clip(test_preds, 1, None)

In [ ]:
# Kaggle submissionfile
submission = pd.DataFrame({
    "id": test_ids,
    "Rings": test_preds
})

submission.to_csv("lasso_submission.csv", index=False)

submission.head()

In [ ]:
# Coefficient Importance
import matplotlib.pyplot as plt

top_coef = coef_df.head(10).sort_values("Absolute Coefficient")

plt.figure(figsize=(8, 6))
plt.barh(top_coef["Feature"], top_coef["Absolute Coefficient"])
plt.xlabel("Absolute Coefficient")
plt.ylabel("Feature")
plt.title("Top 10 Feature Coefficients - Lasso Regression")
plt.show()

In [ ]:
# Actual Vs Predicted Chart
plt.figure(figsize=(7, 7))
plt.scatter(y_valid, valid_preds, alpha=0.4)
plt.xlabel("Actual Rings")
plt.ylabel("Predicted Rings")
plt.title("Actual vs Predicted Rings - Lasso Regression")
plt.plot([y_valid.min(), y_valid.max()], [y_valid.min(), y_valid.max()])
plt.show()